<a href="https://colab.research.google.com/github/Shahrukh2016/LangChain_LangGraph_LangSmith_Revision/blob/main/1_Revision_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!sudo apt-get update -qq && sudo apt-get install -y zstd -qq
!curl -fsSLs https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!ollama pull llama3.2:1b

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ol

In [3]:
!ollama ls

NAME           ID              SIZE      MODIFIED               
llama3.2:1b    baf6a787fdff    1.3 GB    Less than a second ago    


In [5]:
!pip install -q langchain langchain-huggingface langchain-ollama langchain-community langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
## For chat conversions
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model= "llama3.2:1b",
    temperature= 0
)

response = llm.invoke("Why south pole is icy")
print(response)

In [4]:
from google.colab import userdata
# userdata.get('HF_TOKEN')
# userdata.get('HUGGINGFACEHUB_API_TOKEN')

### 1. Text Generation

In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

load_dotenv()

# 1. Use a standard Llama model that accepts the text-generation route flawlessly
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.7
)

# 2. Pass it directly to ChatHuggingFace
model = ChatHuggingFace(llm=llm)

# 3. Invoke the chat model wrapper
result = model.invoke("Name random 5 indian female names?")

# 4. Print the clean text output
print(result.content)

Here are five random Indian female names:

1. **Aarohi** (means "ascending" or "melody")
2. **Meera** (a traditional name, also associated with the poet-saint Meera Bai)
3. **Anaya** (means "grace" or "divine")
4. **Sia** (a modern, short, and sweet name)
5. **Ishani** (means "goddess Durga" or "night")

Would you like names from a specific region or language? 😊


### 2. Embedding Generation

In [19]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv

load_dotenv()

# Defining text
text = 'Hi , Machine Learning is impressive!'

# Initializing embedding model
embedding = HuggingFaceEndpointEmbeddings(model= 'sentence-transformers/all-MiniLM-L6-v2')

# Sending to LLM
result = embedding.embed_query(text=text)

# Printing result
print(result)
print(len(result))

[-0.0026930756866931915, -0.08147257566452026, 0.07546250522136688, -0.006461235228925943, -0.020381951704621315, -0.025554507970809937, -0.07856892794370651, -0.09241099655628204, -0.10938961058855057, -0.0320647656917572, -0.07055692374706268, 0.07044176012277603, 0.06622582674026489, -0.05373391881585121, -0.06714571267366409, 0.009830706752836704, -0.01547983754426241, -0.03206050395965576, -0.11080781370401382, -0.11889339238405228, 0.012000640854239464, -0.010549995116889477, 0.023770686239004135, -0.014055314473807812, 0.03467894345521927, 0.010974934324622154, 0.07937692850828171, 0.04726547747850418, 0.006085897795855999, -0.05813884735107422, 0.01519906334578991, 0.01804661750793457, 0.027096107602119446, 0.016628853976726532, -0.0775386393070221, 0.0460054911673069, -0.0673285648226738, 0.016554737463593483, 0.03365078940987587, 0.01726851612329483, -0.0039319321513175964, -0.03446643054485321, 0.020229317247867584, 0.007600399665534496, 0.07117927819490433, 0.09303563088178

### 3. Document Similarity

In [20]:
## importing necessary libraries
from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

## loading env
load_dotenv()

#######################################################################################################################################################

## A list of technical documents (sentences) to be embedded for similarity comparison
docs = [
    "Neural networks learn to approximate complex functions by adjusting weights through backpropagation.",
    "Docker containers provide a lightweight way to package and deploy applications across environments.",
    "A relational database uses structured query language (SQL) to manage and retrieve data efficiently.",
    "The time complexity of binary search is O(log n), making it faster than linear search for sorted arrays.",
    "Random forest is an ensemble learning method that combines multiple decision trees to improve accuracy.",
    "RESTful APIs use HTTP methods to perform CRUD operations and support stateless communication.",
    "Cloud computing allows scalable, on-demand access to computing resources over the internet.",
    "Principal Component Analysis (PCA) is a dimensionality reduction technique used to capture variance in data.",
    "Version control systems like Git help track changes in code and enable collaborative development.",
    "Anomaly detection algorithms identify data points that significantly deviate from the expected pattern."
]

## The user's search query to find the most relevant document
user_query = "What algorithms are being used in Deep Learning?"

In [22]:
### Initialize the HuggingFace embedding model (MiniLM)
embedding = HuggingFaceEndpointEmbeddings(model= 'sentence-transformers/all-MiniLM-L6-v2')

### Generate embeddings for all the documents
doc_embeeding = embedding.embed_documents(docs)

### Generate embedding for the user's query
query_embeeding = embedding.embed_query(user_query)

In [31]:
cosine_similarity([query_embeeding], doc_embeeding)[0]

array([0.37915375, 0.06532606, 0.1570393 , 0.1345426 , 0.19599357,
       0.11075473, 0.13850883, 0.12008292, 0.1101437 , 0.11210791])

In [33]:
scores = cosine_similarity([query_embeeding], doc_embeeding)[0]
list(enumerate(scores))

[(0, np.float64(0.37915375489859915)),
 (1, np.float64(0.0653260550825593)),
 (2, np.float64(0.1570392988558033)),
 (3, np.float64(0.13454259891404088)),
 (4, np.float64(0.19599357262168807)),
 (5, np.float64(0.11075473115922646)),
 (6, np.float64(0.1385088270925256)),
 (7, np.float64(0.12008291931246487)),
 (8, np.float64(0.11014370265717333)),
 (9, np.float64(0.11210791012387833))]

In [35]:
sorted(list(enumerate(scores)), key = lambda x: x[1], reverse=True)[0]

(0, np.float64(0.37915375489859915))

In [1]:
#######################################################################################################################################################

# Compute cosine similarity between the query embedding and all document embeddings
scores = cosine_similarity([query_embeeding], doc_embeeding)[0]

# Find the document index with the highest similarity score
index, score = sorted(list(enumerate(scores)), key = lambda x: x[1], reverse=True)[0]

# Print the most relevant document along with the similarity score
print(f"The maching document of the user query: '{user_query}' is: '{docs[index]}' with the similarity score of '{score}'")

NameError: name 'cosine_similarity' is not defined

### 4. Chatbot

In [21]:
## Loading libraries and building creating model instace
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


model = ChatOllama(
    model= "llama3.2:1b",
    temperature= 0
)

In [25]:
## Basic chatbot
while True:
  user_input = input('You:' )
  if user_input == 'exit':
    break
  else:
    result = model.invoke(user_input)
    print(f'AI: {result.content}')

You:Choose 2 numbers between 1 to 10
AI: I'll choose two random numbers between 1 and 10. My choices are... 

Number 1: 7
Number 2: 4
You:Can you tell me what are the numbers ?
AI: This conversation has just started. I'm happy to chat with you, but I don't have any context or information about specific numbers yet. If you'd like to share some numbers or ask a question about them, I'll do my best to help!
You:exit


In [24]:
## Chatbot with retaining chat
chat_history = []
while True:
  user_input = input('You:' )
  chat_history.append(user_input)
  if user_input == 'exit':
    break
  else:
    result = model.invoke(chat_history)
    chat_history.append(result.content)
    print(f'AI: {result.content}')

You:Choose 2 numbers between 1 to 10
AI: I'll choose two random numbers between 1 and 10. My choices are... 

Number 1: 8
Number 2: 5
You:Can you tell me what are the numbers ?
AI: The numbers you've chosen are 8 and 5.
You:exot
AI: I can't help with that request.
You:exit


In [ ]:
#### Chatbot with sender identification and retaining chat
chat_history = [
  SystemMessage(content = 'You are a helpful AI Assistant'),
]
while True:
  user_input = input('You:' )
  chat_history.append(HumanMessage(content = user_input))
  if user_input == 'exit':
    break
  else:
    result = model.invoke(chat_history)
    chat_history.append(AIMessage(content = result.content))
    print(f'AI: {result.content}')

print(chat_history)
